# Introduction

## Overview
This notebook serves as a template for analyzing and comparing dimensionality reduction methods used in the experiments, focusing on how well they preserve the dataset's features neighborhood structure. It is designed to work seamlessly with different datasets. Once the evaluation (see `process_dataset.sh`, `evaluation.sh`, and related files) is complete, the notebook automatically generates and presents the result

## Details
The notebook includes a concise, automatically generated analysis to guide subsequent exploration. **One may expand this content** if deeper investigation into the generated graphs and evaluation outcomes is needed. It highlights the best alignment between the methods' graphs and the reference graph based on *Precision*, *Recall*, and *F1-score*, along with descriptive statistics considering all comparisions between method's graph and reference's graph.

# Outset Steps


Here, we define the notebook configurations and the parameters required during execution (e.g., --parameters passed through `critdd_template.ipynb`).

In [ ]:
dataset = None

In [ ]:
if dataset is None:
    raise ValueError("No dataset provided!")

### Essential setup & dependencies

Since this notebook relies on some first-party modules (like the contents of Utils) and will access multiple files, it's convenient to move the notebook to the project root to simplify file access.

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "."))  

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project Root:", project_root) 

The following Libraries will be required during the notebook's execution. 

In [ ]:
import random
import logging
from pathlib import Path

import pandas as pd
import numpy as np
from critdd import Diagram

from Utils import load_edges

We do configure logging to generate a log file for this notebook. However, since we are using Papermill (refer to the `generatecritdd` function in `evaluation.sh`), this step may be somewhat redundant or "overworry".

In [ ]:
log_filename = Path("logs") / dataset / f"critdd_{dataset}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_filename, mode="w"),  
        logging.StreamHandler(sys.stdout)  # Print to console
    ]
)

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Initial Data Exploration & Overview


In this section, we present a code snippet designed for analyzing the generated graphs. This snippet primarily checks for basic inconsistencies, verifying the correct number of edges in a random graph. However, one may expand the code to inspect futher aspects.

## Analyzing generated graph structures


###  Consistency graph selection

In [ ]:
consistency_path = Path('GeneratedGraphs/Consistency') / dataset
consistency_file = random.choice(os.listdir(consistency_path)) 

consistency_graph = load_edges(consistency_path, consistency_file)

print(f"{consistency_file} generated {len(consistency_graph)} edges")

In [ ]:
consistency_graph

### ICA-based representation graph

In [ ]:
ica_path = Path('GeneratedGraphs/ICA') / dataset
ica_file = random.choice(os.listdir(ica_path)) 

ica_graph = load_edges(ica_path, ica_file)

print(f"{ica_file} generated {len(ica_graph)} edges")

Notice that, since the k-nearest neighbors graph is directed, the total of edges must be k_method * number of rows in the original dataset

In [ ]:
ica_graph

### Isomap-based representation graph

In [ ]:
isomap_path = Path('GeneratedGraphs/Isomap') / dataset
isomap_file = random.choice(os.listdir(isomap_path)) 

isomap_graph = load_edges(isomap_path, isomap_file)

print(f"{isomap_file} generated {len(isomap_graph)} edges")

### LLE-based representation graph

In [ ]:
lle_path = Path('GeneratedGraphs/LLE') / dataset
lle_file = random.choice(os.listdir(lle_path)) 

lle_graph = load_edges(lle_path, lle_file)

print(f"{lle_file} generated {len(lle_graph)} edges")

### PCA-based representation graph

In [ ]:
pca_path = Path('GeneratedGraphs/PCA') / dataset
pca_file = random.choice(os.listdir(pca_path)) 

pca_graph = load_edges(pca_path, pca_file)

print(f"{pca_file} generated {len(pca_graph)} edges")

### Gaussian Random Projection-based representation graph

In [ ]:
random_path = Path('GeneratedGraphs/RandomProjection') / dataset
random_file = random.choice(os.listdir(random_path)) 

random_graph = load_edges(random_path, random_file)

print(f"{random_file} generated {len(random_graph)} edges")

### Laplacian Eigenmaps-based representation graph

In [ ]:
spectral_path = Path('GeneratedGraphs/Spectral') / dataset
spectral_file = random.choice(os.listdir(spectral_path)) 

spectral_graph = load_edges(spectral_path, spectral_file)

print(f"{spectral_file} generated {len(spectral_graph)} edges")

### t-SNE-based representation graph

In [ ]:
tsne_path = Path('GeneratedGraphs/TSNE') / dataset
tsne_file = random.choice(os.listdir(tsne_path)) 

tsne_graph = load_edges(tsne_path, tsne_file)

print(f"{tsne_file} generated {len(tsne_graph)} edges")

In [ ]:
tsne_pca_path = Path('GeneratedGraphs/TSNE_PCA') / dataset
tsne_pca_file = random.choice(os.listdir(tsne_pca_path)) 

tsne_pca_graph = load_edges(tsne_pca_path, tsne_pca_file)

print(f"{tsne_pca_file} generated {len(tsne_pca_graph)} edges")

### UMAP-based representation graph

In [ ]:
umap_path = Path('GeneratedGraphs/UMAP') / dataset
umap_file = random.choice(os.listdir(umap_path)) 

umap_graph = load_edges(umap_path, umap_file)

print(f"{umap_file} generated {len(umap_graph)} edges")

# Performance Metrics & Evaluation

In this section, we provide a code snippet that offers a high-level overview of all the graphs generated by the methods. This snippet allows one to quickly inspect and evaluate key properties of each graph, offering a "big picture" perspective.

### Utility functions and variable definition


In [ ]:
def f1_score(precision: float, recall: float)-> float:
  """
  Calculates the F1-score for given precision and recall.

  Parameters
  ----------
  precision : float
      The respective precision value.
  recall : float
      The respective recall value.

  Returns
  -------
  float
      The computed F1-score based on precision and recall.
  """
  return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

def find_best_compared_graphs(df: pd.DataFrame, metric: str) -> tuple[dict[str: list[str]], set[str]]:
    """
    Finds the best graph of the corresponding method for each consistency graph.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe containing the comparisons between method's graphs and consistency graphs.
    metric : str
        The metric to decide which graph was the best match.

    Returns
    -------
    Tuple[Dict[str, List[str]], Set[str]]
        A dictionary mapping each consistency graph to its best matches among method's graphs, and a set containing all unique 
        best-matching graphs.
    """
    
    # Find best matches
    best_graphs = (
        df.loc[df.groupby("Reference Graph")[metric].idxmax(), ["Reference Graph", "Compared Graph"]]
        .groupby("Reference Graph")["Compared Graph"]
        .apply(list)
        .to_dict()
    )

    # Extract unique winners
    unique_winners = set(graph for graphs in best_graphs.values() for graph in graphs)
    
    return best_graphs, unique_winners

def get_highest_scores(method: pd.DataFrame, score: str = "F1_score") -> pd.DataFrame:
    """
    Retrieves the highest-scoring graphs for each reference graph.

    Parameters
    ----------
    method : pd.DataFrame
        The DataFrame containing the scores for different graphs.
    score : str, optional (default="F1_score")
        The column name representing the score metric to compare.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the best-performing graphs for each reference graph.
    """
    # Compute highest scores 
    highest_scores = (
        method.loc[method.groupby("Reference Graph")[score].idxmax()]
        .set_index("Reference Graph")
    )

    return highest_scores

def get_mean_of_top_k_percent(method: pd.DataFrame, score: str = "F1_score", k: float = 10) -> pd.DataFrame:
    """
    Retrieves the mean of the top k% scoring graphs for each reference graph.

    Parameters
    ----------
    method : pd.DataFrame
        The DataFrame containing the scores for different graphs.
    score : str, optional (default="F1_score")
        The column name representing the score metric to compare.
    k : float, optional (default=10)
        The top percentage of scores to consider (e.g., 10 for top 10%).

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the mean of the top k% scores for each reference graph.
    """
    top_k_mean = (
        method.groupby("Reference Graph")
        .apply(lambda x: x.nlargest(int(np.ceil(len(x) * (k / 100))), score)[score].mean())
        .reset_index(name=score)
        .set_index("Reference Graph")
    )
    
    return top_k_mean

In the following cells, we define the variables cointaining the results, accordingly to our repository structure.

In [ ]:
evaluation_path = Path('EvaluationResults/') / dataset

In [ ]:
ica = pd.read_parquet(evaluation_path / 'comparison_results_ICA.parquet')
isomap = pd.read_parquet(evaluation_path / 'comparison_results_Isomap.parquet')
lle = pd.read_parquet(evaluation_path / 'comparison_results_LLE.parquet')
pca = pd.read_parquet(evaluation_path / 'comparison_results_PCA.parquet')
random = pd.read_parquet(evaluation_path / 'comparison_results_RandomProjection.parquet')
spectral = pd.read_parquet(evaluation_path / 'comparison_results_Spectral.parquet')
tsne = pd.read_parquet(evaluation_path / 'comparison_results_TSNE.parquet')
tsne_pca = pd.read_parquet(evaluation_path / 'comparison_results_TSNE+PCA.parquet')
umap = pd.read_parquet(evaluation_path / 'comparison_results_UMAP.parquet')

## ICA

In [ ]:
ica

In [ ]:
ica['F1_score'] = ica.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
ica

In [ ]:
ica.describe()

In [ ]:
ica[ica.Precision == ica.Precision.max()]

In [ ]:
ica[ica.Recall == ica.Recall.max()]

In [ ]:
ica[ica.F1_score == ica.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(ica, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## Isomap

In [ ]:
isomap

In [ ]:
isomap['F1_score'] = isomap.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
isomap

In [ ]:
isomap.describe()

In [ ]:
isomap[isomap.Precision == isomap.Precision.max()]

In [ ]:
isomap[isomap.Recall == isomap.Recall.max()]

In [ ]:
isomap[isomap.F1_score == isomap.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(isomap, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## Locally Linear Embedding

In [ ]:
lle

In [ ]:
lle['F1_score'] = lle.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
lle

In [ ]:
lle.describe()

In [ ]:
lle[lle.Precision == lle.Precision.max()]

In [ ]:
lle[lle.Recall == lle.Recall.max()]

In [ ]:
lle[lle.F1_score == lle.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(lle, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## PCA

In [ ]:
pca

In [ ]:
pca['F1_score'] = pca.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
pca

In [ ]:
pca.describe()

In [ ]:
pca[pca.Precision == pca.Precision.max()]

In [ ]:
pca[pca.Recall == pca.Recall.max()]

In [ ]:
pca[pca.F1_score == pca.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(pca, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## Gaussian Random Projection


In [ ]:
random

In [ ]:
random['F1_score'] = random.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
random

In [ ]:
random.describe()

In [ ]:
random[random.Precision == random.Precision.max()]

In [ ]:
random[random.Recall == random.Recall.max()]

In [ ]:
random[random.F1_score == random.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(random, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## Laplacian Eigenmaps

In [ ]:
spectral

In [ ]:
spectral['F1_score'] = spectral.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
spectral

In [ ]:
spectral.describe()

In [ ]:
spectral[spectral.Precision == spectral.Precision.max()]

In [ ]:
spectral[spectral.Recall == spectral.Recall.max()]

In [ ]:
spectral[spectral.F1_score == spectral.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(spectral, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## t-SNE

In [ ]:
tsne

In [ ]:
tsne['F1_score'] = tsne.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
tsne

In [ ]:
tsne.describe()

In [ ]:
tsne[tsne.Precision == tsne.Precision.max()]

In [ ]:
tsne[tsne.Recall == tsne.Recall.max()]

In [ ]:
tsne[tsne.F1_score == tsne.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(tsne, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## t-SNE + PCA

In [ ]:
tsne_pca

In [ ]:
tsne_pca['F1_score'] = tsne_pca.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
tsne_pca

In [ ]:
tsne_pca.describe()

In [ ]:
tsne_pca[tsne_pca.Precision == tsne_pca.Precision.max()]

In [ ]:
tsne_pca[tsne_pca.Recall == tsne_pca.Recall.max()]

In [ ]:
tsne_pca[tsne_pca.F1_score == tsne_pca.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(tsne_pca, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

## UMAP

In [ ]:
umap

In [ ]:
umap['F1_score'] = umap.apply(lambda row: f1_score(row['Precision'], row['Recall']), axis=1)
umap

In [ ]:
umap.describe()

In [ ]:
umap[umap.Precision == umap.Precision.max()]

In [ ]:
umap[umap.Recall == umap.Recall.max()]

In [ ]:
umap[umap.F1_score == umap.F1_score.max()]

In [ ]:
compared_graph_max, winners = find_best_compared_graphs(umap, 'F1_score')

In [ ]:
compared_graph_max

In [ ]:
winners

# Critical Difference Diagrams

In this section we define the best method, accordingly to our analysis, by Critical Difference Diagrams.

In [ ]:
methods = {
    'ica': ica,
    'isomap': isomap,
    'lle': lle,
    'pca': pca,
    'random_projection': random,
    'laplacian_eigenmaps': spectral,
    'tsne': tsne,
    'tsne+pca': tsne_pca,
    'umap': umap,
}

results_dict = {}
for method_name, data in methods.items():
    top_f1 = get_mean_of_top_k_percent(data)
    results_dict[method_name] = top_f1[['F1_score']].rename(columns={'F1_score': method_name})

results = pd.concat(results_dict.values(), axis=1)


In [ ]:
results

In [ ]:
diagram = Diagram( # from critdd package
    results.to_numpy(),
    treatment_names = results.columns,
    maximize_outcome = True
)

In [ ]:
diagram.average_ranks

In [ ]:
diagram.get_groups(alpha=.05, adjustment="holm")

In [ ]:
# export the diagram to a file
diagram.to_file(
    f"CritddResults/critdd_{dataset}.tex",
    alpha = .05,
    adjustment = "holm",
    reverse_x = True,
    axis_options = {"title": "critdd"},
)

In [ ]:
logging.info("critical difference diagrams generated successfully")